# 🏗️ Notebook 1 — Foundation and Prerequisite Validation

This is the first notebook in the Product Finder Custom Use Case workshop.  
Run this before any other notebook.

## What this notebook does
1. Refreshes and rehydrates azd environment values for cloned repos
2. Reuses the existing Citadel spoke from the selected azd environment
3. Loads and validates all required azd environment values
4. Confirms Azure CLI login and active subscription
5. Verifies Citadel Hub and APIM are deployed and reachable
6. Creates the Product Finder model-access APIM product and subscription (for agent model calls via gateway)
7. Persists shared variables for downstream notebooks

## Permissions model for this workshop
- Required: Azure subscription Owner on the deployed Citadel subscription
- Not required: Entra ID app registration or admin permissions for workshop execution
- Persona is simulated via governance context and APIM subscription key in workshop mode

## Prerequisites
- Citadel workshop notebooks 1 to 8 completed successfully
- az login done in this terminal session
- azd installed and authenticated

## Architecture reminder
```
Scenario Notebooks
      │
      ▼ (Ocp-Apim-Subscription-Key + x-user-persona header)
   APIM (Citadel Hub)  ← Workshop mode: subscription key + persona header
      │
      ▼
  Foundry Hosted Orchestrator Agent (pf-orchestrator)
      │
      ├─► pf-contextualizer
      ├─► pf-product-intelligence
      ├─► pf-compatibility          (elevated-risk only)
      ├─► pf-aligner
      └─► pf-sample-request         (external_customer only)
```
All LLM inference calls from agents flow through APIM via the BYO Gateway connection.

## 1. Load Shared Utils And SPOKE Environment

Import helper utilities, load required SPOKE_* values, and compute Product Finder runtime constants used by later steps.

In [ ]:
import subprocess, sys, pathlib, re

# Resolve shared/ folder robustly from current working directory
cwd = pathlib.Path.cwd().resolve()
candidate_paths = [
    cwd / "shared",
    cwd.parent / "shared",
    cwd.parent.parent / "shared",
    cwd.parent.parent.parent / "shared",
    (cwd / "workshop" / "product-finder" / "notebooks").resolve().parent.parent.parent / "shared",
]

shared_path = next((p for p in candidate_paths if (p / "utils.py").exists()), None)
if shared_path is None:
    raise RuntimeError(
        "Cannot locate shared/utils.py. Run this notebook from the repo workspace and ensure shared/utils.py exists."
    )

sys.path.insert(0, str(shared_path))
import utils  # type: ignore

def azd_get(key):
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Missing azd value [{key}]: {r.stderr.strip()}")
    return r.stdout.strip()

def azd_get_optional(key, default=""):
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        return default
    return (r.stdout or "").strip() or default

def run(cmd, ok_msg="", fail_msg=""):
    return utils.run(cmd, ok_msg, fail_msg)

def normalize_version_tag(tag: str, default: str = "v0") -> str:
    # Keep only semantic tags like v1, v2, ...; if none is set yet, treat that as v0.
    match = re.fullmatch(r"v(\d+)", (tag or "").strip(), re.IGNORECASE)
    if not match:
        return default
    return f"v{int(match.group(1))}"

def image_tag_env_key(agent_name: str) -> str:
    return f"PF_IMAGE_TAG_{agent_name.upper().replace('-', '_')}"

# -- Load existing SPOKE values (primary source of truth).
required_keys = [
    "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_LOCATION",
    "SPOKE_RESOURCE_GROUP", "SPOKE_AI_FOUNDRY_ACCOUNT_NAME",
    "SPOKE_AI_FOUNDRY_PROJECT_NAME", "SPOKE_ACR_NAME", "SPOKE_ACR_LOGIN_SERVER",
]

env = {}
for key in required_keys:
    env[key] = azd_get(key)

# -- Product Finder constants (edit here if needed)
PF_MODEL_CONNECTION   = "Product-Finder-DEV-LLM"        # BYO Gateway connection name in Foundry
PF_MODEL_DEPLOYMENT   = "gpt-5.4-mini"                  # Model deployed behind APIM - chose this for best latency vs reasoning tradeoff for a workshop. If better reasoning is required, better to upgrade the model.
PF_MODEL_FULL         = f"{PF_MODEL_CONNECTION}/{PF_MODEL_DEPLOYMENT}"
PF_SUBSCRIPTION_NAME  = "product-finder-workshop"        # APIM subscription name

SPECIALIST_NAMES = [
    "pf-contextualizer",
    "pf-product-intelligence",
    "pf-compatibility",
    "pf-aligner",
    "pf-sample-request",
]
AGENT_IMAGE_TAG_KEYS = {name: image_tag_env_key(name) for name in SPECIALIST_NAMES}
AGENT_CURRENT_TAGS = {
    name: normalize_version_tag(azd_get_optional(env_key, "v0"))
    for name, env_key in AGENT_IMAGE_TAG_KEYS.items()
}

# Derived values
FOUNDRY_PROJECT_ENDPOINT = (
    f"https://{env['SPOKE_AI_FOUNDRY_ACCOUNT_NAME']}.services.ai.azure.com"
    f"/api/projects/{env['SPOKE_AI_FOUNDRY_PROJECT_NAME']}"
)

utils.print_info(f"Subscription:  {env['AZURE_SUBSCRIPTION_ID']}")
utils.print_info(f"Hub RG:        {env['AZURE_RESOURCE_GROUP']}")
utils.print_info(f"Spoke RG:      {env['SPOKE_RESOURCE_GROUP']}")
utils.print_info(f"Foundry:       {env['SPOKE_AI_FOUNDRY_ACCOUNT_NAME']}")
utils.print_info(f"Project:       {env['SPOKE_AI_FOUNDRY_PROJECT_NAME']}")
utils.print_info(f"ACR:           {env['SPOKE_ACR_NAME']}")
utils.print_info(f"Model (APIM):  {PF_MODEL_FULL}")
utils.print_info("Specialist tags (read from azd env, per agent):")
for agent_name in SPECIALIST_NAMES:
    utils.print_info(f"  {agent_name}: {AGENT_CURRENT_TAGS[agent_name]}  ({AGENT_IMAGE_TAG_KEYS[agent_name]})")
utils.print_info(f"Foundry EP:    {FOUNDRY_PROJECT_ENDPOINT}")

## 2. Validate Azure Login And Subscription

Check active Azure account context and ensure the selected subscription matches the azd environment to avoid cross-subscription deployment mistakes.

In [ ]:
# ── 1️⃣  Azure CLI login and subscription check ──────────────────────────────
out = run("az account show -o json", "az account: OK", "az account: FAILED")
if not out.success:
    raise RuntimeError("Not logged in. Run: az login")

acct = out.json_data
utils.print_info(f"User:         {acct['user']['name']}")
utils.print_info(f"Subscription: {acct['id']}")
utils.print_info(f"Tenant:       {acct['tenantId']}")

if acct["id"] != env["AZURE_SUBSCRIPTION_ID"]:
    raise RuntimeError(
        f"Active subscription {acct['id']} does not match azd subscription "
        f"{env['AZURE_SUBSCRIPTION_ID']}. Run: az account set --subscription {env['AZURE_SUBSCRIPTION_ID']}"
    )
utils.print_ok("Subscription matches azd environment.")

## 3. Verify Hub And Spoke Prerequisites

Confirm APIM in the hub and Foundry plus ACR in the existing spoke are present and reachable before generating contract artifacts.

In [ ]:
# -- 2 Verify APIM resource in hub resource group ------------------------------
hub_rg = env["AZURE_RESOURCE_GROUP"]
apim_out = run(
    f"az resource list -g {hub_rg} --resource-type Microsoft.ApiManagement/service -o json",
    "APIM query OK", "APIM query failed" 
)
if not apim_out.success or not apim_out.json_data:
    raise RuntimeError("No APIM resource found in hub resource group. Ensure Citadel hub is deployed.")

apim_name = apim_out.json_data[0]["name"]
utils.print_ok(f"APIM: {apim_name}  (RG: {hub_rg})")

# -- 3 Verify Foundry in existing spoke ---------------------------------------
foundry_out = run(
    f"az resource show -g {env['SPOKE_RESOURCE_GROUP']} "
    f"--name {env['SPOKE_AI_FOUNDRY_ACCOUNT_NAME']} "
    f"--resource-type Microsoft.CognitiveServices/accounts -o json",
    "Foundry account OK", "Foundry account not found" 
)
if not foundry_out.success:
    raise RuntimeError("Spoke Foundry account not found. Confirm the base spoke deployment.")
utils.print_ok(f"Foundry account: {env['SPOKE_AI_FOUNDRY_ACCOUNT_NAME']}")

# -- 4 Verify ACR in existing spoke -------------------------------------------
acr_out = run(
    f"az acr show -n {env['SPOKE_ACR_NAME']} -g {env['SPOKE_RESOURCE_GROUP']} -o json",
    "ACR OK", "ACR not found" 
)
if not acr_out.success:
    raise RuntimeError("Spoke ACR not found.")
utils.print_ok(f"ACR: {env['SPOKE_ACR_NAME']}  ({env['SPOKE_ACR_LOGIN_SERVER']})")

## 4. Generate Model-Access Contract Artifacts

Create Product Finder APIM access-contract files and parameters that target the existing spoke resources and key vault.

In [ ]:
# -- 5 Prepare Product Finder model-access APIM contract artifacts (Citadel-style) --
import pathlib, os, subprocess

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing bicep/infra/citadel-access-contracts/main.bicep")

def azd_get_optional(key: str, default: str = "") -> str:
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        return default
    return (r.stdout or "").strip() or default

repo_root = find_repo_root(pathlib.Path.cwd())
template_file = repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep"

# Keep Product Finder access contract artifacts local to this workshop folder.
pf_contracts_dir = repo_root / "workshop" / "product-finder" / "access-contracts"

MODEL_CONTRACT = {
    "service_code": "LLM",
    "business_unit": "PF",
    "use_case_name": "ProductFinderModelAccess",
    "environment": "DEV",
    "endpoint_secret": "PF_MODEL_ENDPOINT",
    "apikey_secret": "PF_MODEL_KEY",
}

# Fixed model API list for this use case
model_api_ids = ["universal-llm-api"]

# Governance baseline limits for workshop mode (conservative defaults).
MODEL_TPM = 1000
MODEL_TOKEN_QUOTA = 100000
MODEL_TOKEN_QUOTA_PERIOD = "Monthly"

# Use existing SPOKE Key Vault.
keyvault_rg = env.get("SPOKE_RESOURCE_GROUP", "")
keyvault_name = azd_get_optional("SPOKE_KEY_VAULT_NAME", "")
if keyvault_name:
    kv_show_out = run(
        f"az keyvault show -n {keyvault_name} -g {keyvault_rg} -o json",
        "Spoke Key Vault lookup OK",
        "Spoke Key Vault lookup failed"
    )
    if not kv_show_out.success:
        keyvault_name = ""

if not keyvault_name:
    kv_out = run(
        f"az keyvault list -g {keyvault_rg} -o json",
        "Key Vault query OK",
        "Key Vault query failed"
    )
    if not kv_out.success or not kv_out.json_data:
        raise RuntimeError("No Key Vault found in spoke resource group; cannot enable useTargetAzureKeyVault=true.")
    keyvault_name = kv_out.json_data[0]["name"]

folder_name = f"{MODEL_CONTRACT['business_unit'].lower()}-{MODEL_CONTRACT['use_case_name'].lower()}"
env_folder = MODEL_CONTRACT["environment"].lower()
contract_folder = pf_contracts_dir / "contracts" / folder_name / env_folder
contract_folder.mkdir(parents=True, exist_ok=True)

policy_xml = f'''<policies>
  <inbound>
    <base />
    <include-fragment fragment-id="set-llm-requested-model" />
    <set-variable name="allowedModels" value="{PF_MODEL_DEPLOYMENT}" />
    <include-fragment fragment-id="validate-model-access" />
    <set-variable name="enableResponseHeaders" value="@(true)" />
  </inbound>
  <backend>
    <base />
  </backend>
  <outbound>
    <base />
  </outbound>
  <on-error>
    <base />
  </on-error>
</policies>'''
(contract_folder / "ai-product-policy.xml").write_text(policy_xml, encoding="utf-8")

using_path = pathlib.Path(os.path.relpath(template_file, contract_folder)).as_posix()

# Foundry connection name in the template is: <connectionNamePrefix>-<service_code>.
model_connection_prefix = PF_MODEL_CONNECTION
suffix = f"-{MODEL_CONTRACT['service_code']}"
if model_connection_prefix.endswith(suffix):
    model_connection_prefix = model_connection_prefix[:-len(suffix)]

params_content = f"""using '{using_path}'

param apim = {{
  subscriptionId: '{env['AZURE_SUBSCRIPTION_ID']}'
  resourceGroupName: '{hub_rg}'
  name: '{apim_name}'
}}

param keyVault = {{
  subscriptionId: '{env['AZURE_SUBSCRIPTION_ID']}'
  resourceGroupName: '{keyvault_rg}'
  name: '{keyvault_name}'
}}

param useTargetAzureKeyVault = true

param useCase = {{
  businessUnit: '{MODEL_CONTRACT['business_unit']}'
  useCaseName: '{MODEL_CONTRACT['use_case_name']}'
  environment: '{MODEL_CONTRACT['environment']}'
}}

param apiNameMapping = {{
  {MODEL_CONTRACT['service_code']}: [{', '.join([f"'{api}'" for api in model_api_ids])}]
}}

param services = [
  {{
    code: '{MODEL_CONTRACT['service_code']}'
    endpointSecretName: '{MODEL_CONTRACT['endpoint_secret']}'
    apiKeySecretName: '{MODEL_CONTRACT['apikey_secret']}'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Product Finder model access contract (generated by Notebook 1)'

param useTargetFoundry = true
param foundry = {{
  subscriptionId: '{env['AZURE_SUBSCRIPTION_ID']}'
  resourceGroupName: '{env['SPOKE_RESOURCE_GROUP']}'
  accountName: '{env['SPOKE_AI_FOUNDRY_ACCOUNT_NAME']}'
  projectName: '{env['SPOKE_AI_FOUNDRY_PROJECT_NAME']}'
}}

param foundryConfig = {{
  connectionNamePrefix: '{model_connection_prefix}'
  connectionCategory: 'ApiManagement'
  deploymentInPath: 'false'
  isSharedToAll: false
  inferenceAPIVersion: '2025-03-01-preview'
  deploymentAPIVersion: ''
  staticModels: []
  listModelsEndpoint: ''
  getModelEndpoint: ''
  deploymentProvider: ''
  customHeaders: {{}}
  authConfig: {{}}
}}
"""
params_file = contract_folder / "main.bicepparam"
params_file.write_text(params_content, encoding="utf-8")
utils.print_info(f"Contract artifacts written to: {contract_folder}")
utils.print_info(f"Using Key Vault: {keyvault_name} (RG: {keyvault_rg})")
utils.print_ok("Contract artifacts prepared. Run the next cell to deploy and persist runtime config to azd env.")

## 5. Initialize APIM Client Context

Initialize APIM helper tooling so subscription data can be read immediately after contract deployment.

In [ ]:
# ── 6️⃣  Initialize APIMClientTool (run before deployment) ────────────────────
required_vars = [
    "hub_rg",
    "apim_name",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing variables from previous cell: {missing}. Run Cell 5 first.")

# Match Citadel validation notebooks: initialize APIM client in a dedicated step.
from apimtools import APIMClientTool  # type: ignore[reportMissingImports]

apimClientTool = APIMClientTool(hub_rg, apim_name)
apimClientTool.initialize()
utils.print_ok("APIMClientTool initialized.")
utils.print_info("Run the next cell to deploy the model contract, retrieve the subscription key, and persist runtime config to azd env.")

## 6. Deploy Contract And Persist Runtime Config

Deploy the contract, retrieve the generated subscription key, and save the runtime values into azd env for downstream notebooks.

In [ ]:
# ── 7️⃣  Deploy model-access contract, fetch key, and persist runtime config ──
import datetime, subprocess

required_vars = [
    "template_file",
    "params_file",
    "MODEL_CONTRACT",
    "model_api_ids",
    "keyvault_name",
    "keyvault_rg",
    "apim_name",
    "hub_rg",
    "env",
    "PF_MODEL_FULL",
    "PF_MODEL_CONNECTION",
    "PF_MODEL_DEPLOYMENT",
    "FOUNDRY_PROJECT_ENDPOINT",
    "apimClientTool",
    "SPECIALIST_NAMES",
    "AGENT_IMAGE_TAG_KEYS",
    "AGENT_CURRENT_TAGS",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing variables from previous cell: {missing}. Run Cell 6 first.")

def set_azd_env(key: str, value: str):
    p = subprocess.run(["azd", "env", "set", key, str(value)], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(p.stderr or p.stdout).strip()}")

deployment_name = f"pf-model-contract-{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M%S')}"
deploy_cmd = (
    f"az deployment sub create --name {deployment_name} "
    f"--location {env['AZURE_LOCATION']} "
    f"--template-file \"{template_file}\" "
    f"--parameters \"{params_file}\" -o json"
 )
deploy_out = run(deploy_cmd, "Model contract deployed", "Model contract deployment failed")
if not deploy_out.success:
    err = getattr(deploy_out, "stderr", "") or getattr(deploy_out, "stdout", "")
    raise RuntimeError(f"Model contract deployment failed: {err}")

model_product_id = f"{MODEL_CONTRACT['service_code']}-{MODEL_CONTRACT['business_unit']}-{MODEL_CONTRACT['use_case_name']}-{MODEL_CONTRACT['environment']}"
model_subscription_name = f"{model_product_id}-SUB-01"

# Refresh APIM subscriptions after deployment so newly created subscription key is available.
apimClientTool.initialize()

model_sub_key = ""
for sub in apimClientTool.apim_subscriptions:
    if sub.get("name") == model_subscription_name:
        model_sub_key = sub.get("key", "")
        break

if model_sub_key:
    utils.print_ok(f"Model subscription key retrieved (first 8 chars): {model_sub_key[:8]}...")
else:
    utils.print_warning("Could not retrieve model subscription key. Retrieve it manually if needed.")

# Persist runtime config to azd env for downstream notebooks.
set_azd_env("PF_HUB_RG", hub_rg)
set_azd_env("PF_APIM_NAME", apim_name)
set_azd_env("PF_KEYVAULT_NAME", keyvault_name)
set_azd_env("PF_KEYVAULT_RG", keyvault_rg)
set_azd_env("PF_MODEL_CONNECTION", PF_MODEL_CONNECTION)
set_azd_env("PF_MODEL_DEPLOYMENT", PF_MODEL_DEPLOYMENT)
set_azd_env("PF_MODEL_FULL", PF_MODEL_FULL)
for agent_name in SPECIALIST_NAMES:
    set_azd_env(AGENT_IMAGE_TAG_KEYS[agent_name], AGENT_CURRENT_TAGS[agent_name])
set_azd_env("FOUNDRY_PROJECT_ENDPOINT", FOUNDRY_PROJECT_ENDPOINT)
set_azd_env("PF_MODEL_APIM_PRODUCT_ID", model_product_id)
set_azd_env("PF_MODEL_SUBSCRIPTION_NAME", model_subscription_name)
set_azd_env("PF_MODEL_SUBSCRIPTION_KEY", model_sub_key)
set_azd_env("PF_MODEL_API_IDS", ",".join(model_api_ids))

print()
utils.print_ok("✅ Foundation checks and model-access contract complete. Runtime config saved to azd env. Proceed to Notebook 2.")